## Milestone 2

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

In [2]:
# --- 1. NOVÝ DATASET ČTOUCÍ Z CSV ---
class WikiArtCSVDataset(Dataset):
    def __init__(self, csv_file, split, transform=None):
        # Načtení celého CSV
        self.df = pd.read_csv(csv_file)
        # Filtrace pouze pro požadovaný split (train, val, nebo test)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.transform = transform
        
        # Dynamické vytvoření mapování textových labelů na čísla pro zachovaných 14 tříd
        self.classes = sorted(self.df['label'].unique())
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        label_str = self.df.loc[idx, 'label']
        label = self.class_to_idx[label_str]
        
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [3]:
# --- 2. DYNAMICKÁ AUGMENTACE (DATA ENHANCEMENT) ---
# Aplikuje se pouze na trénovací data pro zamezení overfittingu
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet standard
    transforms.RandomErasing(p=0.2) # Strukturální regularizace
])

# Pro validaci a testování nesmí probíhat náhodné deformace
val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
# --- 3. INICIALIZACE DATALOADERŮ ---
csv_path = "dataset_split.csv"

train_dataset = WikiArtCSVDataset(csv_file=csv_path, split='train', transform=train_transform)
val_dataset = WikiArtCSVDataset(csv_file=csv_path, split='val', transform=val_test_transform)
test_dataset = WikiArtCSVDataset(csv_file=csv_path, split='test', transform=val_test_transform)

print(f"Počet trénovacích vzorků: {len(train_dataset)}")
print(f"Počet validačních vzorků: {len(val_dataset)}")
print(f"Počet testovacích vzorků: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

Počet trénovacích vzorků: 60729
Počet validačních vzorků: 7591
Počet testovacích vzorků: 7592


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používám zařízení: {device}")

# 1. TRANSFER LEARNING: Načtení předtrénovaného modelu
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)


# 2. Zmrazení základních vrstev (frozen base layers)
# Tímto zakážeme výpočet gradientů pro staré vrstvy, aby si síť nezničila užitečné extraktory hran a textur.
for param in model.parameters():
    param.requires_grad = False

# 3. Výměna finálního klasifikátoru
# Nová vrstva má automaticky requires_grad=True, takže se bude trénovat pouze ona.
num_ftrs = model.fc.in_features
num_classes = len(train_dataset.classes) # 14 tříd
model.fc = nn.Linear(num_ftrs, num_classes)

model = model.to(device)

# 4. POKROČILÁ OPTIMALIZACE A REGULARIZACE
# Výběr pouze těch parametrů, které vyžadují gradient (naše nová fc vrstva)
trainable_params = [p for p in model.parameters() if p.requires_grad]

# AdamW obsahuje vylepšenou implementaci L2 regularizace (Weight Decay)
optimizer = optim.AdamW(trainable_params, lr=1e-3, weight_decay=1e-4)

# Dynamické plánování learning ratu pomocí OneCycleLR
# Max_lr je maximální hodnota, na kterou scheduler vystoupá, než začne klesat k nule.
epochs = 10
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,
    steps_per_epoch=len(train_loader),
    epochs=epochs
)

print(f"Model připraven. Počet trénovatelných vrstev: 1 (přepsaná model.fc)")

Používám zařízení: cuda
Model připraven. Počet trénovatelných vrstev: 1 (přepsaná model.fc)


In [6]:
import time
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

# Inicializace TensorBoardu pro logování křivek (Experiment tracking)
writer = SummaryWriter('runs/wikiart_resnet18_frozen')
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0

print("Spouštím trénování s TensorBoard logováním...")
for epoch in range(epochs):
    start_time = time.time()
    
    # --- TRÉNOVACÍ FÁZE ---
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for inputs, labels in train_loop:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping - ochrana proti explodujícím gradientům[cite: 2]
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step() # Krok OneCycleLR plánovače po každém batchi[cite: 2]
        
        running_train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels).item()
        total_train += labels.size(0)
        
        train_loop.set_postfix(loss=running_train_loss/total_train)
        
    epoch_train_loss = running_train_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # --- VALIDAČNÍ FÁZE ---
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]  ")
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels).item()
            total_val += labels.size(0)
            
    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    # Logování do TensorBoardu[cite: 2]
    writer.add_scalar('Loss/Train', epoch_train_loss, epoch)
    writer.add_scalar('Loss/Validation', epoch_val_loss, epoch)
    writer.add_scalar('Accuracy/Train', epoch_train_acc, epoch)
    writer.add_scalar('Accuracy/Validation', epoch_val_acc, epoch)
    
    # Checkpointing - uložení nejlepších vah modelu
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), 'best_resnet18_frozen.pth')
        saved_msg = " -> Uloženo (Nový rekord!)"
    else:
        saved_msg = ""
        
    time_elapsed = time.time() - start_time
    print(f"Shrnutí {epoch+1} [{time_elapsed:.0f}s]: Train Acc: {epoch_train_acc*100:.2f}% | Val Acc: {epoch_val_acc*100:.2f}%{saved_msg}\n")

writer.close()

Spouštím trénování s TensorBoard logováním...


Epoch 1/10 [Val]  : 100%|██████████| 238/238 [01:17<00:00,  3.07it/s]


Shrnutí 1 [610s]: Train Acc: 30.51% | Val Acc: 40.01% -> Uloženo (Nový rekord!)



Epoch 2/10 [Val]  : 100%|██████████| 238/238 [01:17<00:00,  3.06it/s]


Shrnutí 2 [608s]: Train Acc: 37.50% | Val Acc: 39.55%



Epoch 3/10 [Val]  : 100%|██████████| 238/238 [01:17<00:00,  3.09it/s]


... (11 outputs left out)

Epoch 9/10 [Val]  : 100%|██████████| 238/238 [01:17<00:00,  3.05it/s]


Shrnutí 9 [611s]: Train Acc: 42.02% | Val Acc: 44.53% -> Uloženo (Nový rekord!)



Epoch 10/10 [Val]  : 100%|██████████| 238/238 [01:17<00:00,  3.07it/s]

Shrnutí 10 [608s]: Train Acc: 42.58% | Val Acc: 45.21% -> Uloženo (Nový rekord!)



In [ ]:
# 1. Načtení těch nejlepších vah z první zmrazené fáze
model.load_state_dict(torch.load('best_resnet18_frozen.pth'))
print("Nejlepší váhy z 1. fáze úspěšně načteny.")

# 2. Postupné rozmrazení (Gradual unfreezing)
# Rozmrazíme pouze poslední dva konvoluční bloky ResNetu.
# Nejnižší vrstvy (detekce čar a základních tvarů) necháme dál zmrazené.
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

# 3. Diskriminativní learning rate
# Různé části sítě se budou učit různou rychlostí
optimizer = optim.AdamW([
    {'params': model.layer3.parameters(), 'lr': 1e-5}, # Velmi opatrný LR pro starší vrstvy
    {'params': model.layer4.parameters(), 'lr': 5e-5}, 
    {'params': model.fc.parameters(), 'lr': 1e-4}      # Standardní LR pro klasifikační hlavu
], weight_decay=1e-4)

# Znovu vytvoříme scheduler pro dalších 10 epoch
epochs_unfrozen = 35
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[1e-4, 5e-4, 1e-3], # Maxima pro jednotlivé skupiny parametrů
    steps_per_epoch=len(train_loader),
    epochs=epochs_unfrozen
)

# Přepneme logování do nové složky pro TensorBoard
writer = SummaryWriter('runs/wikiart_resnet18_unfrozen')
best_val_acc_unfrozen = 0.0

print("Spouštím 2. fázi: Fine-tuning s rozmrazenými bloky...")
# Následuje naprosto stejná trénovací smyčka jako prve:
for epoch in range(epochs_unfrozen):
    start_time = time.time()
    
    # --- TRÉNOVACÍ FÁZE ---
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs_unfrozen} [Train]")
    for inputs, labels in train_loop:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        running_train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels).item()
        total_train += labels.size(0)
        
        train_loop.set_postfix(loss=running_train_loss/total_train)
        
    epoch_train_loss = running_train_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # --- VALIDAČNÍ FÁZE ---
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs_unfrozen} [Val]  ")
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels).item()
            total_val += labels.size(0)
            
    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    writer.add_scalar('Loss/Train_Unfrozen', epoch_train_loss, epoch)
    writer.add_scalar('Loss/Validation_Unfrozen', epoch_val_loss, epoch)
    writer.add_scalar('Accuracy/Train_Unfrozen', epoch_train_acc, epoch)
    writer.add_scalar('Accuracy/Validation_Unfrozen', epoch_val_acc, epoch)
    
    if epoch_val_acc > best_val_acc_unfrozen:
        best_val_acc_unfrozen = epoch_val_acc
        torch.save(model.state_dict(), 'best_resnet18_final.pth')
        saved_msg = " -> Uloženo (Nový rekord!)"
    else:
        saved_msg = ""
        
    time_elapsed = time.time() - start_time
    print(f"Shrnutí Fáze 2 - {epoch+1} [{time_elapsed:.0f}s]: Train Acc: {epoch_train_acc*100:.2f}% | Val Acc: {epoch_val_acc*100:.2f}%{saved_msg}\n")

writer.close()

/tmp/ipykernel_8045/2558594133.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_resnet18_final.pth'))


Nejlepší váhy z 1. fáze úspěšně načteny.
Spouštím 2. fázi: Fine-tuning s rozmrazenými bloky...


Epoch 1/10 [Train]:  16%|█▌        | 305/1898 [01:28<07:59,  3.32it/s, loss=0.988]Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a5b61fbcd60><function _MultiProcessingDataLoaderIter.__del__ at 0x7a5b61fbcd60>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/pavel/vsb/4/wise/ML_WikiArt_project/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
  File "/home/pavel/vsb/4/wise/ML_WikiArt_project/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/home/pavel/vsb/4/wise/ML_WikiArt_project/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
  File "/home/pavel/vsb/4/wise/ML_WikiArt_project/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
        if w.is_alive():if w.is_al

Shrnutí Fáze 2 - 1 [620s]: Train Acc: 64.34% | Val Acc: 64.80% -> Uloženo (Nový rekord!)



Epoch 2/10 [Train]:  15%|█▌        | 289/1898 [01:23<08:16,  3.24it/s, loss=1.04]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a5b61fbcd60>
Traceback (most recent call last):
  File "/home/pavel/vsb/4/wise/ML_WikiArt_project/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/pavel/vsb/4/wise/ML_WikiArt_project/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
    Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7a5b61fbcd60> 
 Traceback (most recent call last):
   File "/home/pavel/vsb/4/wise/ML_WikiArt_project/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
      sel

KeyboardInterrupt: 